# Kiểm tra dữ liệu

In [1]:
import pandas as pd
from os.path import join

Data_path = join("..","Data","letter_recognition","letter-recognition.data")
cols = ["letter","x-box","y-box","width","high","onpix","x-bar","y-bar", "x2bar",
        "y2bar","xybar","x2ybr","xy2br","x-ege","xegvy","y-ege","yegvx"]
data_raw = pd.read_csv(Data_path, sep =",", header= None, names= cols)
print(data_raw.head())

print(data_raw.info())

  letter  x-box  y-box  width  high  onpix  x-bar  y-bar  x2bar  y2bar  xybar  \
0      T      2      8      3     5      1      8     13      0      6      6   
1      I      5     12      3     7      2     10      5      5      4     13   
2      D      4     11      6     8      6     10      6      2      6     10   
3      N      7     11      6     6      3      5      9      4      6      4   
4      G      2      1      3     1      1      8      6      6      6      6   

   x2ybr  xy2br  x-ege  xegvy  y-ege  yegvx  
0     10      8      0      8      0      8  
1      3      9      2      8      4     10  
2      3      7      3      7      3      9  
3      4     10      6     10      2      8  
4      5      9      1      7      5     10  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   letter  20000 non-null  object
 1   x-box   20000 n

# Mã hóa label (bởi vì K-Nearest phải tính khoảng cách euclid nên không sài chuỗi được)

In [2]:
from sklearn.preprocessing import LabelEncoder

label_Encoder = LabelEncoder()
data_raw['letter'] = label_Encoder.fit_transform(data_raw['letter'])
data_raw.head()

,letter,x-box,y-box,width,high,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
0,19,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,8,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,3,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,13,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,6,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


# Tách data ra chuẩn bị cho việc train và dự đoán

In [3]:
from sklearn.model_selection import train_test_split

X = data_raw.iloc[:,:-1].values
y = data_raw.iloc[:,-1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, random_state= 42, shuffle= True
)

data = {}  
for features, label in zip(X_train, y_train):
    if label not in data:
        data[label] = []
    data[label].append(features)

# Sử dụng hàm predict theo tham khảo

In [5]:
from collections import Counter
import numpy as np

def predict(input_feature_set, k):  
  distances = []

  for group in data:  
    for training_feature_set in data[group]:  
      euclidean_distance = np.linalg.norm(np.array(input_feature_set) - np.array(training_feature_set))  
      distances.append([euclidean_distance, group])

  nearest = sorted(distances)[:k]  
  votes = [d[1] for d in nearest]   
  prediction = Counter(votes).most_common(1)[0][0]

  return prediction

correct = 0
for features, true_label in zip(X_test, y_test):
    pred = predict(features, k=5)
    if pred == true_label:
        correct += 1

accuracy = correct / len(X_test)
print("Accuracy:", accuracy)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
for k, v in data.items():
    print(k, "has", len(v), "samples in training set")

Accuracy: 0.68225
Train size: 16000
Test size: 4000
8 has 6451 samples in training set
7 has 2760 samples in training set
6 has 1474 samples in training set
9 has 1873 samples in training set
11 has 693 samples in training set
10 has 1270 samples in training set
5 has 796 samples in training set
3 has 99 samples in training set
4 has 384 samples in training set
13 has 40 samples in training set
12 has 108 samples in training set
2 has 24 samples in training set
1 has 13 samples in training set
14 has 12 samples in training set
15 has 2 samples in training set
0 has 1 samples in training set
